# Scrape Data

Reverse engineering Yale's energy usage website

In [1]:
import gzip
import json
import re
from io import BytesIO

import polars as pl
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

## Setup

In [2]:
# Make initial request
response = requests.get("https://java.facilities.yale.edu/energy/")
soup = BeautifulSoup(response.text, "html.parser")

# Find script tag and parse out hidden JSONs
script_text = soup.find("script", string=re.compile("var user")).string # type: ignore
user_json = re.search(r'var user\s*=\s*(\{.*?\});', script_text, re.S).group(1) # type: ignore
buildings_json = re.search(r'var buildings\s*=\s*(\[.*?\]);', script_text, re.S).group(1) # type: ignore
user = json.loads(user_json)
buildings = json.loads(buildings_json)

# Save session ID and token to use later
JSESSIONID = user["jsessionID"]
TOKEN = user["token"]

## Buildings/Facilities

In [3]:
# Parse sets
rows = []
for s in user["sets"]:
    set_meta = {
        "bsID": s["bsID"],
        "bsName": s["bsName"],
        "bsDescription": s.get("bsDescription"),
        "bsPublic": s["bsPublic"],
        "owner": s["owner"],
        "sort": s["sort"]
    }

    for b in s["buildings"]:
        row = {**set_meta, **b}
        rows.append(row)

df = pl.DataFrame(rows)

# Extract out facilities, building sets, and set membership
facilities = (
    df
    .unique(subset=["facid"])
    .select([
        "facid",
        "site",
        "name",
        "address",
        "zone",
        "owned",
        "status",
        "latitude",
        "longitude"
    ])
    .sort("facid")
)
building_sets = (
    df
    .unique(subset=["bsID"])
    .select([
        "bsID",
        "bsName",
        "bsDescription",
        "bsPublic",
        "owner",
        "sort"
    ])
    .sort("bsID")
)
set_membership = (
    df
    .select(["facid", "bsID"])
    .unique()
    .sort(["facid", "bsID"])
)

# Save to CSV
facilities.write_csv("../data/raw/facilities.csv")
building_sets.write_csv("../data/raw/building_sets.csv")
set_membership.write_csv("../data/raw/set_membership.csv")

## Facility Details

In [4]:
# Get all facility IDs
facids = facilities["facid"].to_list()

# Set up POST request
cookies = {
    "JSESSIONID": JSESSIONID,
}
headers = {
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'Origin': 'https://java.facilities.yale.edu',
    'Referer': 'https://java.facilities.yale.edu/energy/',
    'X-Requested-With': 'XMLHttpRequest',
}

# Fetch data and save (compressed) JSON
all_results = []
batch_size = 20

for i in tqdm(range(0, len(facids), batch_size)):
    batch = facids[i:i + batch_size]
    facids_str = json.dumps(batch, separators=(',', ':'))

    data = {
        "action": "getUsageSummaryUsageResponseJSON",
        "facids": facids_str
    }

    response = requests.post(
        "https://java.facilities.yale.edu/energy/jsonProvider.jsp",
        headers=headers,
        data=data,
        cookies=cookies,
    )
    all_results.append(response.json())

with gzip.open("../data/raw/facility_details.json.gz", "wt", encoding="utf-8") as f:
    json.dump(all_results, f)

100%|██████████| 19/19 [01:52<00:00,  5.93s/it]


## Energy Usage

In [5]:
# Set up POST request
cookies = {
    "JSESSIONID": JSESSIONID,
}
headers = {
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'Origin': 'https://java.facilities.yale.edu',
    'Referer': 'https://java.facilities.yale.edu/energy/',
    'X-Requested-With': 'XMLHttpRequest',
}

# Fetch data and save (compressed) JSON
all_results = []

for facid in tqdm(facids):
    data = {
        'action': 'runUsageRequestJSON',
        'usageRequest': json.dumps({
            "facids": [facid],
            "startDate": "2000-07-01T07:00:00.000Z",
            "endDate": "2026-07-01T08:00:00.000Z",
            "chartType": "OverallMonthly",
            "calendarType": 0,
            "usageType": 3
        }, separators=(',', ':'))
    }

    response = requests.post(
        "https://java.facilities.yale.edu/energy/jsonProvider.jsp",
        headers=headers,
        data=data,
        cookies=cookies,
    )
    all_results.append(response.json())

with gzip.open("../data/raw/energy_usage.json.gz", "wt", encoding="utf-8") as f:
    json.dump(all_results, f)

100%|██████████| 370/370 [04:58<00:00,  1.24it/s]


## Excel Exports

Dirtier version of energy data above, but has energy per square foot, which we can back out into square footage for facilities that didn't have it listed in the facility details data.

In [6]:
cookies = {
    "JSESSIONID": JSESSIONID,
    "token": TOKEN,
}

dfs = []
for facid in tqdm(facids):
    url = (
        'https://java.facilities.yale.edu/energy/queryresultsdownload?'
        f'facids=["{facid}"]&startDate="1900-01-01T10:00:00.000Z"&'
        'endDate="2026-07-01T11:00:00.000Z"&style=series&commodityCode=all'
    )

    try:
        r = requests.get(url, cookies=cookies)
        r.raise_for_status()

        if not r.headers.get("content-type", "").startswith("application/vnd"):
            print(f"{facid}: request returned HTML or error, skipping")
            continue

        df = pl.read_excel(BytesIO(r.content), infer_schema_length=0)

        df = df.with_columns(
            pl.lit(facid).alias("facid")
        ).select(["facid", *df.columns])

        if "Usage Month" in df.columns and df.height > 0:
            dfs.append(df)

    except Exception as e:
        print(f"{facid}: error — {e}")
        continue

excel_exports: pl.DataFrame = pl.concat(dfs, how="vertical_relaxed")
excel_exports.write_parquet("../data/raw/excel_exports.parquet")

100%|██████████| 370/370 [06:22<00:00,  1.03s/it]
